# Cards retrieval from MUL

In [ ]:
# @title
print("LOADING ANCIENT SCRIPTS")
!python -m pip install jupyter-ui-poll
!python -m pip install fpdf2
import ipywidgets as widgets
import requests
import urllib.parse
import time
import os
from datetime import datetime
from IPython.display import Image, display
from fpdf import FPDF
from ipywidgets import Button
from jupyter_ui_poll import ui_events
from google.colab import output
output.clear()
print("ANCIENT SCRIPTS READY")

In [ ]:
# @title
СПИСОК_МЕХОВ = ""
editor = widgets.Textarea(
    value=СПИСОК_МЕХОВ,
    placeholder='Вставьте список юнитов...',
    layout={'width': '100%', 'height': '400px'},
    spellcheck=False
)

top_m = widgets.IntText(
    value=6,
    description='ОТСТУП_СВЕРХУ:',
    style={'description_width': 'initial'}
)

left_m = widgets.IntText(
    value=6,
    description='ОТСТУП_СЛЕВА:',
    style={'description_width': 'initial'}
)
button = widgets.Button(description="СОХРАНИТЬ И ПРОДОЛЖИТЬ", button_style='danger', layout={'width': '100%'})
output = widgets.Output()

display(editor, widgets.HBox([top_m, left_m]), button, output)

clicked = False
def on_button_clicked(b):
    global clicked
    clicked = True

button.on_click(on_button_clicked)

with ui_events() as poll:
    while not clicked:
        poll(10)
        time.sleep(0.1)

СПИСОК_МЕХОВ = editor.value
ОТСТУП_СВЕРХУ = top_m.value
ОТСТУП_СЛЕВА = left_m.value

with output:
    print("NEXT")


Connect and get confirmation

In [ ]:
# @title

base_url = "https://masterunitlist.azurewebsites.net/Unit/QuickList?Name="
raw_items = [item.strip() for item in СПИСОК_МЕХОВ.replace('"', "").replace("'", "").split(',') if item.strip()]

unit_ids = []
skills = []
cache = {}

for item in raw_items:
    if '^' in item:
        parts = item.split('^')
        name = parts[0].strip()
        skill = parts[-1].strip()
    else:
        name = item.strip()
        skill = ""

    if "Игрок" in name:
        url = name
        unit_id = name
        print(f"{name}")
    else:
        encoded_name = name.replace(' ', '%20')
        url = base_url + encoded_name

        if url in cache:
            unit_id = cache[url]
            status_text = f"Repeat ID {unit_id}"
        else:
            try:
                response = requests.get(url)
                response.raise_for_status()
                data = response.json()

                if data.get("Units") and len(data["Units"]) > 0:
                    unit_id = data["Units"][0]["Id"]
                    cache[url] = unit_id
                    status_text = f"ID {unit_id}"
                else:
                    unit_id = None
                    cache[url] = None
                    status_text = "NOT FOUND"
            except Exception as e:
                unit_id = None
                status_text = f"ERROR: {e}"


            time.sleep(0.1)

        print(f"{url} {status_text} {name} SKILL: {skill}")

    unit_ids.append(unit_id)
    skills.append(skill)

print("\nFinal Unit IDs:", unit_ids)

Make a list of mech cards

In [ ]:
# @title
folder_name = "units_cards"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

base_card_url = "http://masterunitlist.info/Unit/Card/"

images_paths = []

for i, unit_id in enumerate(unit_ids):

    if "Игрок" in str(unit_id):
      print(unit_id)
      continue

    file_path = os.path.join(folder_name, f"card_{unit_id}_{skills[i]}.png")
    img_url = f"{base_card_url}{unit_id}?skill={skills[i]}"

    try:
        response = requests.get(img_url, stream=True)
        response.raise_for_status()

        if "image" in response.headers.get("Content-Type", ""):

            with open(file_path, "wb") as f:
                f.write(response.content)

            images_paths.append(file_path)
            print(f"URL: {img_url}")
            print(f"SAVED: {file_path}")

            display(Image(file_path, width=400))
        else:
            print(f"ID {unit_id}: IMAGE NOT FOUND")

    except Exception as e:
        print(f"ERROR {unit_id}: {e}")

    time.sleep(0.1)

print(f"CARDS: {len(images_paths)}")

Save PDF (DOWNLOAD FILE BUTTON ON THE LEFT PANNEL)

In [ ]:
# @title
def create_as_pdf(unit_ids, output_filename=None):
    if output_filename is None:
        output_filename = f"AlphaStrike Force {datetime.today().strftime('%Y-%m-%d')}.pdf"

    pdf = FPDF(orientation='P', unit='mm', format='A4')
    pdf.add_page()

    pdf.set_font("Helvetica", style="B", size=16)

    card_w = 95
    card_h = 68
    margin_left = ОТСТУП_СЛЕВА
    margin_top = ОТСТУП_СВЕРХУ
    spacing_x = 1
    spacing_y = 1

    current_x = margin_left
    current_y = margin_top

    for unit_id in unit_ids:
        u_id_str = str(unit_id)

        if "Игрок" in u_id_str or "Player" in u_id_str:
            if current_x > margin_left:
                current_y += card_h + spacing_y
                current_x = margin_left

            if current_y + 15 + card_h > 280:
                pdf.add_page()
                current_y = margin_top

            display_text = u_id_str.upper().replace("ИГРОК", "PLAYER")

            pdf.set_xy(margin_left, current_y)
            pdf.set_font("Helvetica", "B", 12)
            pdf.cell(card_w * 2, 6, text=display_text, align='L', new_x="LMARGIN", new_y="NEXT")

            current_y = pdf.get_y() + 2
            current_x = margin_left
            continue

        if current_y + card_h > 295:
            pdf.add_page()
            current_y = margin_top
            current_x = margin_left

        img_path = f"units_cards/card_{u_id_str}.png"

        if os.path.exists(img_path):
            pdf.image(img_path, x=current_x, y=current_y, w=card_w, h=card_h)
        else:
            pdf.set_draw_color(200)
            pdf.rect(current_x, current_y, card_w, card_h)
            pdf.set_font("Helvetica", "", 10)
            pdf.set_xy(current_x, current_y + (card_h/2))
            pdf.cell(card_w, 10, text=f"MISSING: {u_id_str}", align='C')

        pdf.set_draw_color(0)
        pdf.rect(current_x, current_y, card_w, card_h)

        if current_x == margin_left:
            current_x = margin_left + card_w + spacing_x
        else:
            current_x = margin_left
            current_y += card_h + spacing_y

    pdf.output(output_filename)
    print(f"PDF SAVED: {output_filename}, YOU CAN DOWNLOAD THE FILE")

create_as_pdf(unit_ids)